# Kili: Qwen2.5-7B (QLoRA) teacher with 5-fold cross-validation on a Colab GPU

Goal: predict every review of `full_train.csv` with a model that has not seen it (out-of-fold),
to find the wrong labels. Same split as `src/teacher_oof.py` (roberta).

**Before running:** Runtime → Change runtime type → **T4 GPU**.

1. Cell 1 (install), then cell 2: **authorise Google Drive** and upload `full_train.csv`
   (only once: it is then kept in `MyDrive/kili_oof/`).
2. Cell 3: `FOLDS` = the folds to run in this session. Start with `[0]` alone: the time per
   fold is printed at the end (estimated 2 to 3 h on a T4, reviews average 353 tokens).
   Folds already on Drive are skipped: if the session drops, just run everything again.
3. At the end, download `MyDrive/kili_oof/qwen7b_f*.npz`, put the files in
   `kili_data_centric/data/oof/`, then run `python src/relabel_teacher.py qwen7b`.


In [ ]:
!pip -q install -U "transformers>=4.45,<4.47" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=0.34" scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive, files
import os, shutil
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kili_oof"
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{OUT}/full_train.csv"):
    for name in files.upload():
        shutil.move(name, f"{OUT}/{name}")
print(sorted(os.listdir(OUT)))

In [ ]:
FOLDS = [0]  # folds to run in this session (folds already on Drive are skipped)
MODEL = "Qwen/Qwen2.5-7B"
EPOCHS = 1
LR = 1e-4
BATCH = 4
ACCUM = 4
HEAD, TAIL = 128, 384    # start + end of the review (512 tokens)

In [ ]:
import gc, time, numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorWithPadding, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Exactly as in src/teacher_oof.py: same cleaning, same split
train = pd.read_csv(f"{OUT}/full_train.csv", index_col=0)
texts = train.review.str.replace("<br />", " ").tolist()
y = train.category.values
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(texts, y))

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
ids_all = tok(texts, add_special_tokens=False)["input_ids"]
ids_all = [i if len(i) <= HEAD + TAIL else i[:HEAD] + i[-TAIL:] for i in ids_all]
print("tokens: median", int(np.median([len(i) for i in ids_all])), "· truncated",
      np.mean([len(i) == HEAD + TAIL for i in ids_all]).round(3))

def ds(idx):
    return Dataset.from_dict({"input_ids": [ids_all[i] for i in idx],
                              "attention_mask": [[1] * len(ids_all[i]) for i in idx],
                              "label": y[idx].tolist()})

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

for k in FOLDS:
    dest = f"{OUT}/qwen7b_f{k}.npz"
    if os.path.exists(dest):
        print(f"fold {k} already done, skipped"); continue
    tr, va = folds[k]
    t0 = time.time()
    set_seed(k)  # before creating the model (randomly initialised head)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=2, quantization_config=bnb, torch_dtype=torch.float16, device_map={"": 0})
    model.config.pad_token_id = tok.pad_token_id
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, task_type="SEQ_CLS",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    args = TrainingArguments(
        output_dir="out", num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=8, gradient_accumulation_steps=ACCUM,
        warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01,
        fp16=True, group_by_length=True, logging_steps=25, save_strategy="no", report_to="none", seed=k)
    trainer = Trainer(model=model, args=args, train_dataset=ds(tr), data_collator=DataCollatorWithPadding(tok))
    trainer.train()
    # transformers >= 4.46: group_by_length also sorts the data at prediction time -> turn it off
    trainer.args.group_by_length = False
    logits = trainer.predict(ds(va)).predictions
    p = torch.from_numpy(logits).float().softmax(-1)[:, 1].numpy()
    np.savez(dest, idx=va, p=p)
    print(f"fold {k}: accuracy vs noisy labels = {((p > .5) == y[va]).mean():.4f} "
          f"(roberta fold 0: 0.6785) · {(time.time() - t0) / 60:.0f} min -> {dest}")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
print(sorted(f for f in os.listdir(OUT) if f.endswith(".npz")))